# MLB Hitting Data (2021–2025)

This notebook pulls **player-level MLB hitting season totals** from the public MLB Stats API and combines seasons **2021, 2022, 2023, 2024, and 2025** into one dataset.


## Data source and API details

- **Source:** MLB Stats API (public)
- **Endpoint:** `https://statsapi.mlb.com/api/v1/stats`
- **Query parameters:**
  - `stats=season` (season totals)
  - `group=hitting` (batting stats)
  - `sportId=1` (MLB)
  - `season` in `2021..2025`
  - `limit` and `offset` for pagination
- **Python dependencies:** `pandas`, `requests`
- **Output file:** `mlb_hitting_2021_2025.csv`

The resulting schema includes identifiers and common batting fields:
`season`, `player_id`, `player_name`, `team_id`, `team_name`, `games_played`, `at_bats`, `runs`, `hits`, `doubles`, `triples`, `home_runs`, `runs_batted_in`, `walks`, `strikeouts`, `stolen_bases`, `caught_stealing`, `batting_average`, `on_base_percentage`, `slugging_percentage`, `ops`.


In [ ]:
from pathlib import Path
from typing import Dict, List

import pandas as pd
import requests

API_URL = "https://statsapi.mlb.com/api/v1/stats"
SEASONS = [2021, 2022, 2023, 2024, 2025]
LIMIT = 1000
TIMEOUT = (5, 30)  # connect, read
HEADERS = {
    "User-Agent": "ML-Grad-Project/1.0 (data collection notebook)",
    "Accept": "application/json",
}

OUTPUT_CSV = Path("mlb_hitting_2021_2025.csv")


In [ ]:
def fetch_season_hitting_data(season: int, limit: int = LIMIT) -> List[Dict]:
    """Fetch all MLB player hitting season totals for one season."""
    offset = 0
    all_splits: List[Dict] = []
    total_splits_expected = None

    while True:
        params = {
            "stats": "season",
            "group": "hitting",
            "sportId": 1,
            "season": season,
            "limit": limit,
            "offset": offset,
        }

        try:
            response = requests.get(API_URL, params=params, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
        except requests.RequestException as exc:
            raise RuntimeError(
                f"Failed to retrieve MLB hitting data for season {season} at offset {offset}: {exc}"
            ) from exc

        try:
            payload = response.json()
        except ValueError as exc:
            raise RuntimeError(
                f"Received non-JSON response for season {season} at offset {offset}."
            ) from exc

        stats_blocks = payload.get("stats")
        if not isinstance(stats_blocks, list) or not stats_blocks:
            raise RuntimeError(
                f"Unexpected response format for season {season}, offset {offset}: missing 'stats'"
            )

        block = stats_blocks[0]
        splits = block.get("splits")
        if splits is None:
            raise RuntimeError(
                f"Unexpected response format for season {season}, offset {offset}: missing 'splits'"
            )

        if total_splits_expected is None:
            total_splits_expected = int(block.get("totalSplits", 0))
            if total_splits_expected <= 0:
                raise RuntimeError(
                    f"No hitting data reported by the API for season {season}; aborting."
                )

        all_splits.extend(splits)

        if len(all_splits) >= total_splits_expected:
            break

        if not splits:
            raise RuntimeError(
                f"Pagination ended early for season {season}. "
                f"Expected {total_splits_expected} rows, got {len(all_splits)}."
            )

        offset += len(splits)

    if total_splits_expected is not None and len(all_splits) != total_splits_expected:
        raise RuntimeError(
            f"Incomplete data for season {season}. "
            f"Expected {total_splits_expected} rows, got {len(all_splits)}."
        )

    return all_splits


def normalize_splits_to_rows(splits: List[Dict], season: int) -> List[Dict]:
    rows = []
    for split in splits:
        player = split.get("player", {})
        team = split.get("team", {})
        stat = split.get("stat", {})

        rows.append(
            {
                "season": season,
                "player_id": player.get("id"),
                "player_name": player.get("fullName"),
                "team_id": team.get("id"),
                "team_name": team.get("name"),
                "games_played": stat.get("gamesPlayed"),
                "at_bats": stat.get("atBats"),
                "runs": stat.get("runs"),
                "hits": stat.get("hits"),
                "doubles": stat.get("doubles"),
                "triples": stat.get("triples"),
                "home_runs": stat.get("homeRuns"),
                "runs_batted_in": stat.get("rbi"),
                "walks": stat.get("baseOnBalls"),
                "strikeouts": stat.get("strikeOuts"),
                "stolen_bases": stat.get("stolenBases"),
                "caught_stealing": stat.get("caughtStealing"),
                "batting_average": stat.get("avg"),
                "on_base_percentage": stat.get("obp"),
                "slugging_percentage": stat.get("slg"),
                "ops": stat.get("ops"),
            }
        )
    return rows


In [ ]:
all_rows = []

for season in SEASONS:
    splits = fetch_season_hitting_data(season)
    if not splits:
        raise RuntimeError(f"No hitting rows returned for season {season}; aborting.")
    all_rows.extend(normalize_splits_to_rows(splits, season))

if not all_rows:
    raise RuntimeError("No rows were returned from the MLB Stats API; aborting CSV export.")

hitting_df = pd.DataFrame(all_rows)

column_order = [
    "season",
    "player_id",
    "player_name",
    "team_id",
    "team_name",
    "games_played",
    "at_bats",
    "runs",
    "hits",
    "doubles",
    "triples",
    "home_runs",
    "runs_batted_in",
    "walks",
    "strikeouts",
    "stolen_bases",
    "caught_stealing",
    "batting_average",
    "on_base_percentage",
    "slugging_percentage",
    "ops",
]

hitting_df = hitting_df[column_order]

numeric_columns = [
    "season",
    "player_id",
    "team_id",
    "games_played",
    "at_bats",
    "runs",
    "hits",
    "doubles",
    "triples",
    "home_runs",
    "runs_batted_in",
    "walks",
    "strikeouts",
    "stolen_bases",
    "caught_stealing",
]

for col in numeric_columns:
    hitting_df[col] = pd.to_numeric(hitting_df[col], errors="coerce")

hitting_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(hitting_df):,} rows to {OUTPUT_CSV.resolve()}")


In [ ]:
row_counts = hitting_df.groupby("season", dropna=False).size().rename("row_count").reset_index()
row_counts


In [ ]:
hitting_df.head(10)
